In [0]:
from pyspark.sql import functions as F

BASE = '/Volumes/northbridge/raw/staging'

raw = (spark.read.option('header', True).option('inferSchema', True)
       .csv(f'{BASE}/PS_20174392719_1491204439457_log.csv'))

# raw = raw.sample(0.25, seed=7)   # uncomment if the full file feels slow

tx = (raw
  .withColumn('txn_id', F.concat(F.lit('T'),
     F.lpad(F.monotonically_increasing_id().cast('string'), 15, '0')))
  .withColumn('day', F.floor((F.col('step') - 1) / 24).cast('int') + 1))

tx.write.mode('overwrite').parquet(f'{BASE}/tx_master')   # freeze IDs
tx = spark.read.parquet(f'{BASE}/tx_master')
print(tx.count(), tx.select('day').distinct().count())

In [0]:
print(tx.count(), tx.select('txn_id').distinct().count())

In [0]:
dups = tx.sample(0.002, seed=3)

bad_neg = (tx.limit(10)
  .withColumn('txn_id', F.concat(F.lit('TBAD'), F.col('txn_id')))
  .withColumn('amount', -F.abs(F.col('amount'))))

bad_null = (tx.limit(10)
  .withColumn('txn_id', F.lit(None).cast('string')))

gateway = tx.unionByName(dups).unionByName(bad_neg).unionByName(bad_null)

(gateway.repartition('day').write.mode('overwrite')
   .partitionBy('day').option('header', True)
   .csv(f'{BASE}/by_day/transactions'))

In [0]:
r = F.rand(11)
ledger = (tx.withColumn('r', r)
  .filter(F.col('r') > 0.005)                       # ~0.5% missing
  .withColumn('ledger_amount',
      F.when(F.col('r') <= 0.008, F.round(F.col('amount') * 1.01, 2))  # ~0.3% mismatch
       .otherwise(F.col('amount')))
  .select('txn_id', F.col('nameOrig').alias('account_id'),
          F.col('type').alias('txn_type'), 'ledger_amount', 'day'))

(ledger.repartition('day').write.mode('overwrite')
   .partitionBy('day').option('header', True)
   .csv(f'{BASE}/by_day/ledger'))

In [0]:
first  = F.array(*[F.lit(x) for x in ['Aarav','Priya','Rohan','Sneha','Vikram','Ananya','Karan','Meera']])
last   = F.array(*[F.lit(x) for x in ['Sharma','Reddy','Patel','Nair','Singh','Das','Rao','Iyer']])
cities = F.array(*[F.lit(x) for x in ['Mumbai','Delhi','Chennai','Kolkata','Hyderabad','Pune']])

cust = (tx.select(F.col('nameOrig').alias('customer_id')).distinct()
  .withColumn('h', F.abs(F.hash('customer_id')))
  .withColumn('full_name', F.concat_ws(' ',
      F.element_at(first, (F.col('h') % 8) + 1),
      F.element_at(last, ((F.col('h') / 8).cast('int') % 8) + 1)))
  .withColumn('phone', F.concat(F.lit('9'), F.lpad((F.col('h') % 1000000000).cast('string'), 9, '0')))
  .withColumn('city', F.element_at(cities, (F.col('h') % 6) + 1))
  .withColumn('account_status', F.lit('ACTIVE'))
  .withColumn('updated_at', F.lit('2024-01-01 00:00:00').cast('timestamp'))
  .drop('h'))

chg = (cust.sample(0.02, seed=5)
  .withColumn('city', F.when(F.col('city') == 'Mumbai', 'Pune').otherwise('Mumbai'))
  .withColumn('account_status', F.when(F.rand(1) < 0.5, 'DORMANT').otherwise('ACTIVE'))
  .withColumn('updated_at', F.lit('2024-01-15 00:00:00').cast('timestamp')))

cust.write.mode('overwrite').option('header', True).csv(f'{BASE}/by_day/customers/day=1')
chg.write.mode('overwrite').option('header', True).csv(f'{BASE}/by_day/customers/day=15')

In [0]:
for p in ['transactions', 'ledger', 'customers']:
    print(p, [f.name for f in dbutils.fs.ls(f'{BASE}/by_day/{p}')][:5])

g = spark.read.option('header', True).csv(f'{BASE}/by_day/transactions')
print('gateway rows', g.count(), '| distinct txn_id', g.select('txn_id').distinct().count())